# StyleGAN-NADA: качественный Google Colab «фото → аниме»

Полный ноутбук для обучения и инференса StyleGAN-NADA. Эта версия специально исправляет две причины плохих результатов на реальных фотографиях:

1. лицо сначала выравнивается по пяти ключевым точкам, а не просто обрезается по центру;
2. инверсия выполняется в два этапа: сначала устойчивый латент W, затем уточнение W+ и шумов.

Дополнительно обучение ограничено безопасными synthesis-блоками, используется меньший learning rate, регуляризация отклонения весов и защита от высокочастотных артефактов. Для реального фото выводятся три силы стилизации, поэтому можно выбрать наиболее аккуратный вариант.

**Запуск**
1. Выберите `Среда выполнения → Сменить среду выполнения → T4 GPU`.
2. Выполняйте ячейки сверху вниз.
3. Для нового качественного обучения оставьте `run_training=True`.
4. Для быстрой проверки временно поставьте `train_steps=10`.
5. Старые веса с уже испорченной геометрией лучше не использовать: проектор не может полностью исправить повреждённый генератор.

## 1. Установка зависимостей

In [ ]:
# Установка зависимостей и загрузка официального StyleGAN2-ADA.

import os
import sys
import subprocess
from pathlib import Path

ROOT = Path("/content")
SG2_DIR = ROOT / "stylegan2-ada-pytorch"
TORCH_EXTENSIONS_DIR = ROOT / "torch_extensions"
TORCH_EXTENSIONS_DIR.mkdir(parents=True, exist_ok=True)

os.environ["TORCH_EXTENSIONS_DIR"] = str(TORCH_EXTENSIONS_DIR)
os.environ["PYTHONUNBUFFERED"] = "1"

def run_command(command):
    print(" ".join(map(str, command)))
    subprocess.run(command, check=True)

packages = [
    "ninja",
    "pyspng",
    "imageio-ffmpeg",
    "ftfy",
    "regex",
    "tqdm",
    "gdown",
    "scipy",
    "opencv-python-headless",
    "gradio>=5,<7",
]

run_command([sys.executable, "-m", "pip", "install", "-q", *packages])
run_command([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-deps",
    "facenet-pytorch==2.6.0",
])
run_command([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "git+https://github.com/openai/CLIP.git",
])

if not SG2_DIR.exists():
    run_command([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/NVlabs/stylegan2-ada-pytorch.git",
        str(SG2_DIR),
    ])

if str(SG2_DIR) not in sys.path:
    sys.path.insert(0, str(SG2_DIR))

print("Установка завершена.")

## 2. Импорты, GPU и стабильные операции StyleGAN

In [ ]:
# Импорты, проверка GPU, воспроизводимость и исправление CUDA-плагинов.

import copy
import gc
import json
import math
import os
import pickle
import random
import shutil
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import clip
import cv2
import dnnlib
import legacy
import matplotlib.pyplot as plt
import numpy as np
import PIL.Image
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from IPython.display import display
from torch import nn
from torchvision.utils import make_grid
from tqdm.auto import tqdm

from torch_utils import custom_ops
from torch_utils.ops import bias_act, upfirdn2d

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU не обнаружен. В Colab выберите: "
        "Среда выполнения → Сменить среду выполнения → T4 GPU."
    )

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def force_reference_stylegan_ops() -> None:
    custom_ops.verbosity = "none"

    upfirdn2d._plugin = None
    upfirdn2d._inited = True
    upfirdn2d._init = lambda: False

    bias_act._plugin = None
    bias_act._inited = True
    bias_act._init = lambda: False

    warnings.filterwarnings(
        "ignore",
        message=r"Failed to build CUDA kernels for upfirdn2d.*",
    )
    warnings.filterwarnings(
        "ignore",
        message=r"Failed to build CUDA kernels for bias_act.*",
    )

force_reference_stylegan_ops()

assert upfirdn2d._init() is False
assert bias_act._init() is False

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {GPU_NAME}")
print(f"Память GPU: {GPU_MEMORY_GB:.1f} GB")
print("CUDA extensions StyleGAN отключены.")
print("Используется стабильная PyTorch reference-реализация.")

## 3. Настройки эксперимента

На T4 используется разрешение 256. Качественный режим рассчитан примерно на 700 шагов обучения и 700 шагов проекции. Для проверки кода можно временно уменьшить оба значения. Безопасные блоки `32–256` не затрагивают самые грубые уровни `4–16`, отвечающие за основную геометрию головы.

In [ ]:
@dataclass
class Config:
    run_training: bool = True
    model_resolution: int = 256
    train_steps: int = 700
    batch_size: int = 1
    gradient_accumulation: int = 2
    learning_rate: float = 0.001
    min_learning_rate_ratio: float = 0.12
    warmup_steps: int = 40
    top_k_blocks: int = 3
    min_train_resolution: int = 32
    max_train_resolution: int = 256
    refresh_selected_blocks: bool = False
    layer_refresh_every: int = 150
    adaptive_latent_steps: int = 4
    adaptive_latent_lr: float = 0.035
    style_mixing_probability: float = 0.35
    clip_views: int = 6
    clip_crop_min: float = 0.78
    structure_weight: float = 0.16
    artifact_weight: float = 0.08
    weight_preservation_weight: float = 0.0005
    target_margin_weight: float = 0.08
    target_margin: float = 0.08
    grad_clip_norm: float = 0.8
    sample_every: int = 50
    checkpoint_every: int = 100
    sample_truncation: float = 0.78
    fixed_samples: int = 8
    projection_steps: int = 700
    projection_w_steps: int = 180
    projection_w_avg_samples: int = 10000
    inference_strengths: Tuple[float, ...] = (0.55, 0.70, 0.85)
    save_to_google_drive: bool = True
    launch_gradio: bool = True
    seed: int = 42

CFG = Config()

MODEL_URLS = {
    256: (
        "https://nvlabs-fi-cdn.nvidia.com/"
        "stylegan2-ada-pytorch/pretrained/transfer-learning-source-nets/"
        "ffhq-res256-mirror-paper256-noaug.pkl"
    ),
    512: (
        "https://nvlabs-fi-cdn.nvidia.com/"
        "stylegan2-ada-pytorch/pretrained/transfer-learning-source-nets/"
        "ffhq-res512-mirror-stylegan2-noaug.pkl"
    ),
}

if CFG.model_resolution not in MODEL_URLS:
    raise ValueError("Поддерживаются разрешения 256 и 512.")

if CFG.model_resolution == 512 and GPU_MEMORY_GB < 20:
    raise RuntimeError(
        "Для 512 px желательно не менее 20 GB GPU. "
        "На T4 используйте model_resolution=256."
    )

SOURCE_PROMPTS = [
    "a centered front-facing portrait photo of a real person",
    "a realistic studio headshot photograph of a human face",
    "a high quality photographic portrait with natural skin",
    "a symmetrical realistic human face photograph",
]

TARGET_PROMPTS = [
    (
        "a clean polished Japanese anime portrait, coherent facial anatomy, "
        "symmetrical expressive eyes, crisp line art, smooth cel shading, "
        "detailed illustrated hair, simple background"
    ),
    (
        "a professional 2D anime character headshot, balanced face, "
        "clean outlines, controlled cel colors, natural pose, detailed hair"
    ),
    (
        "a high quality manga inspired portrait, coherent eyes and mouth, "
        "smooth unbroken skin, elegant linework, non-photorealistic face"
    ),
    (
        "a centered anime face illustration with clean anatomy, "
        "sharp but smooth line art, expressive eyes and polished shading"
    ),
]

NEGATIVE_PROMPTS = [
    "a photorealistic human portrait",
    "a retouched fashion photograph",
    "a 3D rendered plastic doll",
    "a malformed face with missing eyes",
    "an asymmetrical face with broken anatomy",
    "a noisy face with cracks and black streaks",
    "a distorted portrait with melted facial features",
]

CLIP_SPECS = [("ViT-B/32", 1.0)]
if GPU_MEMORY_GB >= 22:
    CLIP_SPECS.append(("ViT-B/16", 0.55))

OUTPUT_ROOT = Path("/content/stylegan_nada_anime_quality")
MODEL_DIR = OUTPUT_ROOT / "models"
SAMPLE_DIR = OUTPUT_ROOT / "training_samples"
LOG_DIR = OUTPUT_ROOT / "logs"
CACHE_DIR = OUTPUT_ROOT / "cache"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
RESULT_DIR = OUTPUT_ROOT / "photo_results"
INPUT_DIR = OUTPUT_ROOT / "input_photos"
ALIGNED_DIR = OUTPUT_ROOT / "aligned_photos"
FINAL_MODEL_PATH = MODEL_DIR / "stylegan_nada_anime_generator.pkl"

for directory in [
    MODEL_DIR,
    SAMPLE_DIR,
    LOG_DIR,
    CACHE_DIR,
    VALIDATION_DIR,
    RESULT_DIR,
    INPUT_DIR,
    ALIGNED_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RESUME_CHECKPOINT = ""
AUTO_RESUME = True
MODEL_URL = MODEL_URLS[CFG.model_resolution]

print(json.dumps(asdict(CFG), ensure_ascii=False, indent=2))
print("CLIP:", CLIP_SPECS)
print("Результаты:", OUTPUT_ROOT)

## 4. Загрузка исходного и аниме-генераторов

In [ ]:
from google.colab import files

print("Загружается исходный FFHQ StyleGAN2-ADA...")
with dnnlib.util.open_url(MODEL_URL, cache_dir=str(CACHE_DIR)) as file:
    network_data = legacy.load_network_pkl(file)

G_source = network_data["G_ema"].eval().requires_grad_(False).to(DEVICE)

if G_source.img_resolution != CFG.model_resolution:
    raise RuntimeError(
        f"Ожидалось разрешение {CFG.model_resolution}, "
        f"но модель имеет {G_source.img_resolution}."
    )

if CFG.run_training:
    G_anime = copy.deepcopy(G_source).train().requires_grad_(False).to(DEVICE)
    print("Создана обучаемая копия генератора.")
else:
    if not FINAL_MODEL_PATH.exists():
        print("Загрузите stylegan_nada_anime_generator.pkl.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Файл модели не выбран.")
        uploaded_name = next(iter(uploaded))
        shutil.move(uploaded_name, FINAL_MODEL_PATH)

    with FINAL_MODEL_PATH.open("rb") as file:
        payload = pickle.load(file)

    G_anime = payload["G_ema"] if isinstance(payload, dict) else payload
    G_anime = G_anime.eval().requires_grad_(False).to(DEVICE)
    print("Загружена готовая аниме-модель:", FINAL_MODEL_PATH)

with torch.no_grad():
    smoke_z = torch.randn(1, G_source.z_dim, device=DEVICE)
    smoke_ws = G_source.mapping(smoke_z, None, truncation_psi=0.7)
    source_smoke = G_source.synthesis(
        smoke_ws,
        noise_mode="const",
        force_fp32=True,
    )
    anime_smoke = G_anime.synthesis(
        smoke_ws,
        noise_mode="const",
        force_fp32=True,
    )

expected_shape = (
    1,
    3,
    CFG.model_resolution,
    CFG.model_resolution,
)
if tuple(source_smoke.shape) != expected_shape:
    raise RuntimeError(f"Неверная форма изображения: {source_smoke.shape}")
if not torch.isfinite(source_smoke).all() or not torch.isfinite(anime_smoke).all():
    raise RuntimeError("Модель создала NaN или Inf во время smoke test.")

del smoke_z, smoke_ws, source_smoke, anime_smoke
gc.collect()
torch.cuda.empty_cache()

print("Разрешение:", G_source.img_resolution)
print("Размерность Z:", G_source.z_dim)
print("Размерность W:", G_source.w_dim)
print("Количество W-векторов:", G_source.num_ws)
print("Блоки:", list(G_source.synthesis.block_resolutions))
print("Smoke test успешно пройден.")

## 5. CLIP-направление photo → anime

In [ ]:
# Загрузка CLIP и построение текстовых направлений.

CLIP_MEAN = torch.tensor(
    [0.48145466, 0.4578275, 0.40821073],
    device=DEVICE,
).view(1, 3, 1, 1)

CLIP_STD = torch.tensor(
    [0.26862954, 0.26130258, 0.27577711],
    device=DEVICE,
).view(1, 3, 1, 1)

@torch.no_grad()
def encode_text_average(model, prompts: Sequence[str]) -> torch.Tensor:
    tokens = clip.tokenize(list(prompts), truncate=True).to(DEVICE)
    features = model.encode_text(tokens).float()
    features = F.normalize(features, dim=-1)
    features = features.mean(dim=0)
    return F.normalize(features, dim=0)

clip_guides = []

for model_name, model_weight in CLIP_SPECS:
    print(f"Загружается CLIP {model_name}...")
    model, _ = clip.load(
        model_name,
        device=DEVICE,
        jit=False,
        download_root=str(CACHE_DIR / "clip"),
    )
    model.eval().requires_grad_(False)

    source_text = encode_text_average(model, SOURCE_PROMPTS)
    target_text = encode_text_average(model, TARGET_PROMPTS)
    negative_text = torch.stack([
        encode_text_average(model, [prompt])
        for prompt in NEGATIVE_PROMPTS
    ])

    text_direction = F.normalize(target_text - source_text, dim=0)

    clip_guides.append({
        "name": model_name,
        "weight": float(model_weight),
        "model": model,
        "input_resolution": int(model.visual.input_resolution),
        "source_text": source_text,
        "target_text": target_text,
        "negative_text": negative_text,
        "text_direction": text_direction,
    })

print("CLIP-направления подготовлены.")

## 6. Вспомогательные функции, выравнивание лица и функции потерь

Для реальных фотографий используется MTCNN и similarity transform по глазам, носу и уголкам рта. Если лицо не найдено, применяется безопасный центральный crop и выводится предупреждение.

In [ ]:
def image_tensor_to_pil(image: torch.Tensor) -> Image.Image:
    image = image.detach().float().cpu().clamp(-1, 1)
    image = ((image + 1.0) * 127.5).round().to(torch.uint8)
    image = image.permute(1, 2, 0).numpy()
    return Image.fromarray(image)

def center_crop_face(image: Image.Image, resolution: int) -> Image.Image:
    image = ImageOps.exif_transpose(image).convert("RGB")
    return ImageOps.fit(
        image,
        (resolution, resolution),
        method=Image.Resampling.LANCZOS,
        centering=(0.5, 0.42),
    )

def pil_to_model_tensor(image: Image.Image, resolution: int) -> torch.Tensor:
    image = center_crop_face(image, resolution)
    array = np.asarray(image, dtype=np.float32).copy()
    return torch.from_numpy(array).permute(2, 0, 1) / 127.5 - 1.0

_FACE_DETECTOR = None

def get_face_detector():
    global _FACE_DETECTOR
    if _FACE_DETECTOR is None:
        try:
            from facenet_pytorch import MTCNN
            _FACE_DETECTOR = MTCNN(
                keep_all=True,
                device=DEVICE,
                post_process=False,
            )
        except Exception as error:
            warnings.warn(
                f"MTCNN недоступен, будет использован центральный crop: {error}"
            )
            _FACE_DETECTOR = False
    return _FACE_DETECTOR

def align_face_ffhq(
    image: Image.Image,
    resolution: int,
) -> Tuple[Image.Image, Dict]:
    image = ImageOps.exif_transpose(image).convert("RGB")
    detector = get_face_detector()

    if detector is False:
        return center_crop_face(image, resolution), {
            "method": "center_crop",
            "confidence": None,
        }

    try:
        boxes, probabilities, landmarks = detector.detect(
            image,
            landmarks=True,
        )
    except Exception as error:
        warnings.warn(f"Детектор лица завершился с ошибкой: {error}")
        boxes = probabilities = landmarks = None

    if boxes is None or landmarks is None or len(boxes) == 0:
        warnings.warn("Лицо не найдено. Использован центральный crop.")
        return center_crop_face(image, resolution), {
            "method": "center_crop",
            "confidence": None,
        }

    valid = []
    for index, (box, probability) in enumerate(zip(boxes, probabilities)):
        if probability is None or not np.isfinite(probability):
            continue
        width = max(1.0, float(box[2] - box[0]))
        height = max(1.0, float(box[3] - box[1]))
        score = float(probability) * math.sqrt(width * height)
        valid.append((score, index))

    if not valid:
        return center_crop_face(image, resolution), {
            "method": "center_crop",
            "confidence": None,
        }

    _, best_index = max(valid)
    source_points = np.asarray(landmarks[best_index], dtype=np.float32)
    scale = resolution / 256.0
    target_points = np.asarray([
        [82.0, 103.0],
        [174.0, 103.0],
        [128.0, 145.0],
        [96.0, 181.0],
        [160.0, 181.0],
    ], dtype=np.float32) * scale

    matrix, _ = cv2.estimateAffinePartial2D(
        source_points,
        target_points,
        method=cv2.LMEDS,
    )

    if matrix is None:
        warnings.warn("Не удалось вычислить alignment. Использован crop.")
        return center_crop_face(image, resolution), {
            "method": "center_crop",
            "confidence": float(probabilities[best_index]),
        }

    image_array = np.asarray(image)
    aligned = cv2.warpAffine(
        image_array,
        matrix,
        (resolution, resolution),
        flags=cv2.INTER_LANCZOS4,
        borderMode=cv2.BORDER_REFLECT_101,
    )
    aligned = Image.fromarray(aligned)
    return aligned, {
        "method": "mtcnn_5_landmarks",
        "confidence": float(probabilities[best_index]),
    }

def save_tensor_grid(
    images: torch.Tensor,
    path: Path,
    nrow: int,
    padding: int = 2,
) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    grid = make_grid(
        images.detach().float().cpu().clamp(-1, 1),
        nrow=nrow,
        padding=padding,
        normalize=True,
        value_range=(-1, 1),
    )
    image_tensor_to_pil(grid * 2.0 - 1.0).save(path)
    return path

def prepare_paired_clip_views(
    source_images: torch.Tensor,
    target_images: torch.Tensor,
    input_resolution: int,
    num_views: int,
    crop_min: float,
) -> Tuple[torch.Tensor, torch.Tensor]:
    _, _, height, width = source_images.shape
    source_views = []
    target_views = []

    for view_index in range(num_views):
        if view_index == 0:
            crop_height = height
            crop_width = width
            top = 0
            left = 0
        else:
            scale = random.uniform(crop_min, 1.0)
            crop_height = max(48, int(height * scale))
            crop_width = max(48, int(width * scale))
            top = random.randint(0, height - crop_height)
            left = random.randint(0, width - crop_width)

        source_crop = source_images[
            :, :, top:top + crop_height, left:left + crop_width
        ]
        target_crop = target_images[
            :, :, top:top + crop_height, left:left + crop_width
        ]

        if view_index > 0 and random.random() < 0.5:
            source_crop = torch.flip(source_crop, dims=[3])
            target_crop = torch.flip(target_crop, dims=[3])

        source_crop = F.interpolate(
            source_crop,
            size=(input_resolution, input_resolution),
            mode="bicubic",
            align_corners=False,
            antialias=True,
        )
        target_crop = F.interpolate(
            target_crop,
            size=(input_resolution, input_resolution),
            mode="bicubic",
            align_corners=False,
            antialias=True,
        )
        source_views.append(source_crop)
        target_views.append(target_crop)

    source_views = torch.cat(source_views, dim=0)
    target_views = torch.cat(target_views, dim=0)
    source_views = ((source_views + 1.0) / 2.0 - CLIP_MEAN) / CLIP_STD
    target_views = ((target_views + 1.0) / 2.0 - CLIP_MEAN) / CLIP_STD
    return source_views, target_views

def aggregate_view_features(
    features: torch.Tensor,
    num_views: int,
    batch_size: int,
) -> torch.Tensor:
    features = F.normalize(features.float(), dim=-1)
    features = features.view(num_views, batch_size, -1).mean(dim=0)
    return F.normalize(features, dim=-1)

def compute_clip_losses(
    source_images: torch.Tensor,
    target_images: torch.Tensor,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    batch_size = source_images.shape[0]
    total_loss = torch.zeros([], device=DEVICE)
    metrics = {}

    for guide in clip_guides:
        source_views, target_views = prepare_paired_clip_views(
            source_images,
            target_images,
            guide["input_resolution"],
            CFG.clip_views,
            CFG.clip_crop_min,
        )

        with torch.no_grad():
            source_features = guide["model"].encode_image(source_views)
            source_features = aggregate_view_features(
                source_features,
                CFG.clip_views,
                batch_size,
            )

        target_features = guide["model"].encode_image(target_views)
        target_features = aggregate_view_features(
            target_features,
            CFG.clip_views,
            batch_size,
        )
        image_direction = target_features - source_features
        text_direction = guide["text_direction"].unsqueeze(0).expand_as(
            image_direction
        )
        directional_loss = (
            1.0
            - F.cosine_similarity(
                image_direction,
                text_direction,
                dim=-1,
                eps=1e-4,
            )
        ).mean()
        positive_similarity = (
            target_features * guide["target_text"].unsqueeze(0)
        ).sum(dim=-1)
        negative_similarity = (
            target_features @ guide["negative_text"].T
        ).max(dim=-1).values
        margin_loss = F.relu(
            negative_similarity
            - positive_similarity
            + CFG.target_margin
        ).mean()
        model_loss = directional_loss + CFG.target_margin_weight * margin_loss
        total_loss = total_loss + guide["weight"] * model_loss
        metrics[f"{guide['name']}_direction"] = float(
            directional_loss.detach().cpu()
        )
        metrics[f"{guide['name']}_margin"] = float(
            margin_loss.detach().cpu()
        )

    return total_loss, metrics

def low_frequency_structure_loss(
    source_images: torch.Tensor,
    target_images: torch.Tensor,
) -> torch.Tensor:
    source_low = F.interpolate(source_images, size=(32, 32), mode="area")
    target_low = F.interpolate(target_images, size=(32, 32), mode="area")
    return F.l1_loss(target_low, source_low)

def high_frequency_excess_loss(
    source_images: torch.Tensor,
    target_images: torch.Tensor,
) -> torch.Tensor:
    source_blur = F.avg_pool2d(source_images, 3, stride=1, padding=1)
    target_blur = F.avg_pool2d(target_images, 3, stride=1, padding=1)
    source_energy = (source_images - source_blur).abs().mean()
    target_energy = (target_images - target_blur).abs().mean()
    return F.relu(target_energy - 1.35 * source_energy)

def sample_training_ws(batch_size: int) -> torch.Tensor:
    z = torch.randn(batch_size, G_source.z_dim, device=DEVICE)
    with torch.no_grad():
        ws = G_source.mapping(z, None)
        if (
            G_source.num_ws > 1
            and random.random() < CFG.style_mixing_probability
        ):
            z_second = torch.randn(batch_size, G_source.z_dim, device=DEVICE)
            ws_second = G_source.mapping(z_second, None)
            cutoff = random.randint(1, G_source.num_ws - 1)
            ws[:, cutoff:] = ws_second[:, cutoff:]
    return ws

## 7. Поиск безопасных обучаемых параметров

Mapping, affine, ToRGB и грубые блоки `4–16` заморожены. Обучаются только convolution-параметры выбранных блоков `32–256`. Это уменьшает риск расплавленных лиц и потери основной формы головы.

In [ ]:
block_records = []
candidate_parameters = []
candidate_records = []
candidate_parameters_by_resolution = {}
optimizer = None
selected_blocks = []
block_scores = {}

if CFG.run_training:
    source_named_parameters = dict(G_source.named_parameters())
    w_index = 0

    for resolution in G_anime.synthesis.block_resolutions:
        block = getattr(G_anime.synthesis, f"b{resolution}")
        used_ws = list(range(
            w_index,
            min(w_index + block.num_conv + block.num_torgb, G_anime.num_ws),
        ))
        w_index += block.num_conv
        block_candidates = []
        safe_resolution = (
            CFG.min_train_resolution
            <= int(resolution)
            <= CFG.max_train_resolution
        )

        for parameter_name, parameter in block.named_parameters():
            full_name = f"synthesis.b{resolution}.{parameter_name}"
            first_module = parameter_name.split(".")[0]
            is_convolution_parameter = first_module.startswith("conv")
            is_affine_parameter = ".affine." in f".{parameter_name}."
            parameter.requires_grad_(False)

            if (
                safe_resolution
                and is_convolution_parameter
                and not is_affine_parameter
            ):
                block_candidates.append(parameter)
                candidate_parameters.append(parameter)
                candidate_records.append((
                    full_name,
                    parameter,
                    source_named_parameters[full_name].detach(),
                ))

        if safe_resolution and block_candidates:
            candidate_parameters_by_resolution[int(resolution)] = block_candidates
            block_records.append({
                "resolution": int(resolution),
                "block": block,
                "used_ws": used_ws,
            })

    for parameter in G_anime.mapping.parameters():
        parameter.requires_grad_(False)

    if not candidate_parameters:
        raise RuntimeError("Не найдены безопасные параметры для обучения.")

    optimizer = torch.optim.Adam(
        candidate_parameters,
        lr=CFG.learning_rate,
        betas=(0.0, 0.99),
    )

def activate_selected_blocks(selected_resolutions: Sequence[int]) -> int:
    selected_set = set(map(int, selected_resolutions))
    trainable_count = 0
    for resolution, parameters in candidate_parameters_by_resolution.items():
        enabled = resolution in selected_set
        for parameter in parameters:
            parameter.requires_grad_(enabled)
            if enabled:
                trainable_count += parameter.numel()
    return trainable_count

def weight_preservation_loss() -> torch.Tensor:
    losses = [
        F.mse_loss(parameter, source_parameter)
        for _, parameter, source_parameter in candidate_records
        if parameter.requires_grad
    ]
    if not losses:
        return torch.zeros([], device=DEVICE)
    return torch.stack(losses).mean()

if CFG.run_training:
    print("Безопасные блоки:", sorted(candidate_parameters_by_resolution))
    print("Параметров-кандидатов:", sum(p.numel() for p in candidate_parameters))
else:
    print("Обучение отключено: параметры модели остаются замороженными.")

## 8. Адаптивный выбор блоков

Небольшая оптимизация W+ показывает, какие уровни латента сильнее реагируют на направление `photo → anime`. Рейтинг строится только среди безопасных блоков, поэтому грубая геометрия лица остаётся замороженной.

In [ ]:
def encode_single_clip_image(
    guide: Dict,
    images: torch.Tensor,
) -> torch.Tensor:
    images = F.interpolate(
        images,
        size=(guide["input_resolution"], guide["input_resolution"]),
        mode="bicubic",
        align_corners=False,
        antialias=True,
    )
    images = ((images + 1.0) / 2.0 - CLIP_MEAN) / CLIP_STD
    return F.normalize(guide["model"].encode_image(images).float(), dim=-1)

def select_adaptive_blocks() -> Tuple[List[int], Dict[int, float]]:
    guide = clip_guides[0]
    selection_batch = max(1, min(2, CFG.batch_size))
    z = torch.randn(selection_batch, G_source.z_dim, device=DEVICE)

    with torch.no_grad():
        initial_ws = G_source.mapping(z, None)
        initial_images = G_source.synthesis(
            initial_ws,
            noise_mode="const",
            force_fp32=True,
        )
        initial_features = encode_single_clip_image(guide, initial_images)

    optimized_ws = initial_ws.detach().clone().requires_grad_(True)
    latent_optimizer = torch.optim.Adam(
        [optimized_ws],
        lr=CFG.adaptive_latent_lr,
    )

    for _ in range(CFG.adaptive_latent_steps):
        latent_optimizer.zero_grad(set_to_none=True)
        generated = G_source.synthesis(
            optimized_ws,
            noise_mode="const",
            force_fp32=True,
        )
        features = encode_single_clip_image(guide, generated)
        image_direction = features - initial_features
        directional = (
            1.0
            - F.cosine_similarity(
                image_direction,
                guide["text_direction"].unsqueeze(0),
                dim=-1,
                eps=1e-4,
            )
        ).mean()
        target_similarity = (
            features * guide["target_text"].unsqueeze(0)
        ).sum(dim=-1).mean()
        loss = directional + 0.15 * (1.0 - target_similarity)
        loss.backward()
        latent_optimizer.step()

    delta_per_w = (
        optimized_ws.detach() - initial_ws
    ).abs().mean(dim=(0, 2))
    scores = {}

    for record in block_records:
        indices = record["used_ws"]
        score = delta_per_w[indices].mean() if indices else torch.zeros([], device=DEVICE)
        scores[record["resolution"]] = float(score.cpu())

    top_k = min(CFG.top_k_blocks, len(scores))
    selected = [
        resolution
        for resolution, _ in sorted(
            scores.items(),
            key=lambda item: item[1],
            reverse=True,
        )[:top_k]
    ]
    selected.sort()
    return selected, scores

if CFG.run_training:
    selected_blocks, block_scores = select_adaptive_blocks()
    trainable_count = activate_selected_blocks(selected_blocks)
    print("Выбранные блоки:", selected_blocks)
    print("Оценки:", block_scores)
    print("Обучаемых параметров:", f"{trainable_count:,}")

## 9. Фиксированные seed до обучения

In [ ]:
fixed_generator = torch.Generator(device=DEVICE)
fixed_generator.manual_seed(CFG.seed)
FIXED_Z = torch.randn(
    CFG.fixed_samples,
    G_source.z_dim,
    generator=fixed_generator,
    device=DEVICE,
)

with torch.no_grad():
    FIXED_WS = G_source.mapping(
        FIXED_Z,
        None,
        truncation_psi=CFG.sample_truncation,
    )
    source_preview = G_source.synthesis(
        FIXED_WS,
        noise_mode="const",
        force_fp32=True,
    )

preview_path = save_tensor_grid(
    source_preview,
    SAMPLE_DIR / "source_before_training.png",
    nrow=4,
)
display(Image.open(preview_path))

## 10. Checkpoints и предпросмотр

In [ ]:
# Обучение StyleGAN-NADA с автоматическим resume и безопасными checkpoints.

import re

def checkpoint_step_from_path(path: Path) -> int:
    match = re.search(r"checkpoint_(\d+)\.pt$", path.name)
    return int(match.group(1)) if match else -1

def find_resume_checkpoint() -> Optional[Path]:
    if RESUME_CHECKPOINT:
        explicit_path = Path(RESUME_CHECKPOINT)

        if not explicit_path.exists():
            raise FileNotFoundError(
                f"Checkpoint не найден: {explicit_path}"
            )

        return explicit_path

    if not AUTO_RESUME:
        return None

    checkpoints = sorted(
        MODEL_DIR.glob("checkpoint_*.pt"),
        key=checkpoint_step_from_path,
    )

    return checkpoints[-1] if checkpoints else None

def move_optimizer_state_to_device(
    current_optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> None:
    for state in current_optimizer.state.values():
        for key, value in list(state.items()):
            if torch.is_tensor(value):
                state[key] = value.to(device)

def save_checkpoint(
    step: int,
    selected_resolutions: Sequence[int],
    current_history: List[Dict],
) -> Path:
    checkpoint_path = MODEL_DIR / f"checkpoint_{step:04d}.pt"
    temporary_path = checkpoint_path.with_suffix(".tmp")

    payload = {
        "step": int(step),
        "generator_state_dict": G_anime.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "selected_blocks": list(map(int, selected_resolutions)),
        "history": current_history,
        "config": asdict(CFG),
        "source_prompts": SOURCE_PROMPTS,
        "target_prompts": TARGET_PROMPTS,
        "model_url": MODEL_URL,
    }

    torch.save(payload, temporary_path)
    os.replace(temporary_path, checkpoint_path)

    return checkpoint_path

def save_training_preview(step: int) -> Path:
    G_anime.eval()

    with torch.no_grad():
        source = G_source.synthesis(
            FIXED_WS,
            noise_mode="const",
            force_fp32=True,
        )
        target = G_anime.synthesis(
            FIXED_WS,
            noise_mode="const",
            force_fp32=True,
        )

    G_anime.train()

    paired = torch.cat([source, target], dim=0)
    path = SAMPLE_DIR / f"source_anime_{step:04d}.png"

    save_tensor_grid(
        paired,
        path,
        nrow=CFG.fixed_samples,
    )

    return path

## 11. Восстановление состояния обучения

In [ ]:
history = []
start_step = 0
last_completed_step = -1
training_error = None

if CFG.run_training:
    resume_path = find_resume_checkpoint()

    if resume_path is not None:
        print("Найден checkpoint:", resume_path)
        state = torch.load(
            resume_path,
            map_location=DEVICE,
            weights_only=False,
        )
        saved_config = state.get("config", {})

        if (
            saved_config
            and int(saved_config.get(
                "model_resolution",
                CFG.model_resolution,
            )) != CFG.model_resolution
        ):
            raise RuntimeError(
                "Разрешение checkpoint не совпадает с текущей конфигурацией."
            )

        G_anime.load_state_dict(state["generator_state_dict"], strict=True)
        optimizer.load_state_dict(state["optimizer_state_dict"])
        move_optimizer_state_to_device(optimizer, DEVICE)
        selected_blocks = list(state.get("selected_blocks", selected_blocks))
        activate_selected_blocks(selected_blocks)
        history = list(state.get("history", []))
        last_completed_step = int(state["step"])
        start_step = last_completed_step + 1
        print(f"Обучение продолжится с шага {start_step}/{CFG.train_steps}.")
    else:
        print("Checkpoint не найден. Обучение начнётся с шага 0.")
else:
    print("Используется готовая модель. Этап обучения пропущен.")

## 12. Стабильное обучение

Используются gradient accumulation, warmup и cosine decay. Помимо CLIP-направления оптимизируются сохранение низкочастотной структуры, ограничение лишнего высокочастотного шума и близость обучаемых весов к исходному генератору.

In [ ]:
def scheduled_learning_rate(step: int) -> float:
    if step < CFG.warmup_steps:
        warmup = (step + 1) / max(1, CFG.warmup_steps)
        return CFG.learning_rate * warmup
    progress = (
        step - CFG.warmup_steps
    ) / max(1, CFG.train_steps - CFG.warmup_steps - 1)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    ratio = CFG.min_learning_rate_ratio + (
        1.0 - CFG.min_learning_rate_ratio
    ) * cosine
    return CFG.learning_rate * ratio

if CFG.run_training:
    G_source.eval()
    G_anime.train()

    try:
        progress = tqdm(
            range(start_step, CFG.train_steps),
            initial=start_step,
            total=CFG.train_steps,
        )

        for step in progress:
            if (
                CFG.refresh_selected_blocks
                and step > start_step
                and step % CFG.layer_refresh_every == 0
            ):
                selected_blocks, block_scores = select_adaptive_blocks()
                activate_selected_blocks(selected_blocks)

            current_lr = scheduled_learning_rate(step)
            for parameter_group in optimizer.param_groups:
                parameter_group["lr"] = current_lr

            optimizer.zero_grad(set_to_none=True)
            metric_sums = {}
            clip_value = 0.0
            structure_value = 0.0
            artifact_value = 0.0

            for _ in range(CFG.gradient_accumulation):
                ws = sample_training_ws(CFG.batch_size)
                with torch.no_grad():
                    source_images = G_source.synthesis(
                        ws,
                        noise_mode="const",
                        force_fp32=True,
                    )

                target_images = G_anime.synthesis(
                    ws,
                    noise_mode="const",
                    force_fp32=True,
                )
                clip_loss, clip_metrics = compute_clip_losses(
                    source_images,
                    target_images,
                )
                structure_loss = low_frequency_structure_loss(
                    source_images,
                    target_images,
                )
                artifact_loss = high_frequency_excess_loss(
                    source_images,
                    target_images,
                )
                data_loss = (
                    clip_loss
                    + CFG.structure_weight * structure_loss
                    + CFG.artifact_weight * artifact_loss
                )
                (data_loss / CFG.gradient_accumulation).backward()
                clip_value += float(clip_loss.detach().cpu())
                structure_value += float(structure_loss.detach().cpu())
                artifact_value += float(artifact_loss.detach().cpu())
                for key, value in clip_metrics.items():
                    metric_sums[key] = metric_sums.get(key, 0.0) + value

            weight_loss = weight_preservation_loss()
            (CFG.weight_preservation_weight * weight_loss).backward()

            trainable_now = [
                parameter
                for parameter in candidate_parameters
                if parameter.requires_grad
            ]
            torch.nn.utils.clip_grad_norm_(
                trainable_now,
                max_norm=CFG.grad_clip_norm,
            )
            optimizer.step()

            clip_value /= CFG.gradient_accumulation
            structure_value /= CFG.gradient_accumulation
            artifact_value /= CFG.gradient_accumulation
            total_value = (
                clip_value
                + CFG.structure_weight * structure_value
                + CFG.artifact_weight * artifact_value
                + CFG.weight_preservation_weight * float(weight_loss.detach().cpu())
            )

            if not math.isfinite(total_value):
                raise RuntimeError(f"На шаге {step} loss стал нечисловым.")

            record = {
                "step": int(step),
                "learning_rate": float(current_lr),
                "total_loss": float(total_value),
                "clip_loss": float(clip_value),
                "structure_loss": float(structure_value),
                "artifact_loss": float(artifact_value),
                "weight_loss": float(weight_loss.detach().cpu()),
                "selected_blocks": list(map(int, selected_blocks)),
                **{
                    key: value / CFG.gradient_accumulation
                    for key, value in metric_sums.items()
                },
            }
            history.append(record)
            last_completed_step = step
            progress.set_postfix({
                "loss": f"{record['total_loss']:.4f}",
                "lr": f"{current_lr:.2e}",
                "blocks": ",".join(map(str, selected_blocks)),
            })

            if step % CFG.sample_every == 0 or step == CFG.train_steps - 1:
                preview_path = save_training_preview(step)
                print(f"\nРезультат на шаге {step}:")
                display(Image.open(preview_path))

            if (
                step > 0
                and (
                    step % CFG.checkpoint_every == 0
                    or step == CFG.train_steps - 1
                )
            ):
                checkpoint_path = save_checkpoint(step, selected_blocks, history)
                print("\nСохранён checkpoint:", checkpoint_path)

    except KeyboardInterrupt:
        print("\nОбучение остановлено пользователем.")
        if last_completed_step >= 0:
            emergency_path = save_checkpoint(
                last_completed_step,
                selected_blocks,
                history,
            )
            print("Текущий прогресс сохранён:", emergency_path)

    except Exception as error:
        training_error = error
        print("\nВо время обучения возникла ошибка:", repr(error))
        if last_completed_step >= 0:
            emergency_path = save_checkpoint(
                last_completed_step,
                selected_blocks,
                history,
            )
            print("Прогресс перед ошибкой сохранён:", emergency_path)
else:
    print("Обучение не запускалось.")

## 13. Экспорт итогового генератора

In [ ]:
final_checkpoint = None
history_path = LOG_DIR / "training_history.json"

if CFG.run_training:
    if last_completed_step < 0:
        raise RuntimeError(
            "Не выполнено ни одного шага обучения."
        ) from training_error

    final_checkpoint = save_checkpoint(
        last_completed_step,
        selected_blocks,
        history,
    )
    history_path.write_text(
        json.dumps(history, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    G_anime.eval()
    export_generator = (
        copy.deepcopy(G_anime)
        .eval()
        .requires_grad_(False)
        .cpu()
    )
    temporary_pkl = FINAL_MODEL_PATH.with_suffix(".tmp")

    with temporary_pkl.open("wb") as file:
        pickle.dump({
            "G_ema": export_generator,
            "metadata": {
                "completed_step": int(last_completed_step),
                "source_prompts": SOURCE_PROMPTS,
                "target_prompts": TARGET_PROMPTS,
                "model_url": MODEL_URL,
                "config": asdict(CFG),
            },
        }, file)

    os.replace(temporary_pkl, FINAL_MODEL_PATH)
    del export_generator
    gc.collect()
    torch.cuda.empty_cache()
    print("Обучение и сохранение завершены.")
    print("Последний выполненный шаг:", last_completed_step)
    print("Checkpoint:", final_checkpoint)
    print("Модель:", FINAL_MODEL_PATH)

    if training_error is not None:
        raise training_error
else:
    G_anime.eval()
    print("Готовая модель используется без переобучения:", FINAL_MODEL_PATH)

## 14. График обучения

In [ ]:
curves_path = LOG_DIR / "training_curves.png"

if history:
    steps = [item["step"] for item in history]
    total_losses = [item["total_loss"] for item in history]
    clip_losses = [item["clip_loss"] for item in history]
    structure_losses = [item["structure_loss"] for item in history]
    artifact_losses = [item.get("artifact_loss", 0.0) for item in history]

    plt.figure(figsize=(10, 5))
    plt.plot(steps, total_losses, label="Общий loss")
    plt.plot(steps, clip_losses, label="Directional CLIP")
    plt.plot(steps, structure_losses, label="Structure")
    plt.plot(steps, artifact_losses, label="Artifact")
    plt.xlabel("Шаг")
    plt.ylabel("Значение")
    plt.title("Обучение StyleGAN-NADA")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(curves_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("График сохранён:", curves_path)
else:
    print("История обучения отсутствует: график пропущен.")

## 15. Примеры на случайных seed

In [ ]:
G_anime.eval()
generator = torch.Generator(device=DEVICE).manual_seed(2026)
z = torch.randn(
    16,
    G_anime.z_dim,
    generator=generator,
    device=DEVICE,
)

with torch.no_grad():
    ws = G_source.mapping(
        z,
        None,
        truncation_psi=CFG.sample_truncation,
    )
    source_random = G_source.synthesis(
        ws,
        noise_mode="const",
        force_fp32=True,
    )
    anime_random = G_anime.synthesis(
        ws,
        noise_mode="const",
        force_fp32=True,
    )

random_pair_path = save_tensor_grid(
    torch.cat([source_random, anime_random], dim=0),
    SAMPLE_DIR / "random_source_and_anime.png",
    nrow=8,
)
print("Верхняя строка — исходные лица, нижняя — аниме.")
display(Image.open(random_pair_path))

## 16. Валидация на фиксированных seed

- `direction_alignment`: совпадение изменения изображения с направлением photo → anime; больше — лучше.
- `target_similarity`: близость результата к anime-промптам; больше — лучше.
- `source_similarity`: остаточная близость к фотореализму; слишком высокое значение означает слабую стилизацию.
- `structure_l1`: изменение низкочастотной структуры; слишком большое значение означает потерю позы и композиции.

In [ ]:
guide = clip_guides[0]
clip_model = guide["model"]
source_text = guide["source_text"]
target_text = guide["target_text"]
text_direction = guide["text_direction"]

@torch.no_grad()
def encode_validation_images(images: torch.Tensor) -> torch.Tensor:
    resolution = guide["input_resolution"]
    images = F.interpolate(
        images,
        size=(resolution, resolution),
        mode="bicubic",
        align_corners=False,
        antialias=True,
    )
    images = ((images + 1.0) / 2.0 - CLIP_MEAN) / CLIP_STD
    return F.normalize(clip_model.encode_image(images).float(), dim=-1)

validation_generator = torch.Generator(device=DEVICE).manual_seed(CFG.seed)
validation_z = torch.randn(
    CFG.fixed_samples,
    G_source.z_dim,
    generator=validation_generator,
    device=DEVICE,
)

with torch.no_grad():
    validation_ws = G_source.mapping(
        validation_z,
        None,
        truncation_psi=CFG.sample_truncation,
    )
    validation_source = G_source.synthesis(
        validation_ws,
        noise_mode="const",
        force_fp32=True,
    )
    validation_anime = G_anime.synthesis(
        validation_ws,
        noise_mode="const",
        force_fp32=True,
    )
    source_features = encode_validation_images(validation_source)
    anime_features = encode_validation_images(validation_anime)

image_direction = F.normalize(anime_features - source_features, dim=-1)
direction_alignment = F.cosine_similarity(
    image_direction,
    text_direction.unsqueeze(0),
    dim=-1,
).mean()
target_similarity = (
    anime_features * target_text.unsqueeze(0)
).sum(dim=-1).mean()
source_similarity = (
    anime_features * source_text.unsqueeze(0)
).sum(dim=-1).mean()
source_low = F.interpolate(validation_source, size=(32, 32), mode="area")
anime_low = F.interpolate(validation_anime, size=(32, 32), mode="area")
structure_l1 = F.l1_loss(anime_low, source_low)

metrics = {
    "direction_alignment": float(direction_alignment.cpu()),
    "target_similarity": float(target_similarity.cpu()),
    "source_similarity": float(source_similarity.cpu()),
    "structure_l1": float(structure_l1.cpu()),
}
validation_grid = VALIDATION_DIR / "validation_source_anime.png"
metrics_path = VALIDATION_DIR / "validation_metrics.json"
save_tensor_grid(
    torch.cat([validation_source, validation_anime]),
    validation_grid,
    nrow=CFG.fixed_samples,
)
metrics_path.write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Верхняя строка — source, нижняя — anime.")
display(Image.open(validation_grid))

## 17. Подготовка безопасной силы стилизации

Полная адаптация (`1.0`) иногда слишком сильно меняет лицо. Поэтому вычисляется разность весов между `G_source` и `G_anime`, после чего можно применять только часть этой разности. На реальных фотографиях обычно наиболее аккуратны значения `0.55–0.85`.

In [ ]:
for guide_item in clip_guides:
    guide_item["model"].to("cpu")

del source_features, anime_features
del validation_source, validation_anime
gc.collect()
torch.cuda.empty_cache()

source_parameters = dict(G_source.named_parameters())
anime_parameters = dict(G_anime.named_parameters())
style_deltas = {}

with torch.no_grad():
    for name, anime_parameter in anime_parameters.items():
        source_parameter = source_parameters[name]
        if not torch.equal(anime_parameter, source_parameter):
            style_deltas[name] = (
                anime_parameter.detach() - source_parameter.detach()
            ).clone()

G_stylized = copy.deepcopy(G_source).eval().requires_grad_(False).to(DEVICE)
stylized_parameters = dict(G_stylized.named_parameters())

def set_style_strength(strength: float) -> None:
    strength = float(np.clip(strength, 0.0, 1.0))
    with torch.no_grad():
        for name, delta in style_deltas.items():
            stylized_parameters[name].copy_(
                source_parameters[name] + strength * delta
            )

G_anime.to("cpu")
gc.collect()
torch.cuda.empty_cache()
print("CLIP и полная G_anime перенесены на CPU.")
print("Изменённых тензоров генератора:", len(style_deltas))

## 18. Улучшенный двухэтапный W/W+ projector

Сначала оптимизируется один общий вектор W, который устойчиво восстанавливает основную форму лица. Затем он разворачивается в W+ для уточнения деталей. Проектор одновременно использует LPIPS-признаки VGG16 и пиксельную ошибку на уменьшенном изображении, а также сохраняет лучшее найденное состояние вместо последнего.

In [ ]:
class ImprovedWPlusProjector:
    def __init__(
        self,
        generator: nn.Module,
        device: torch.device,
        w_avg_samples: int = 10000,
        statistics_batch_size: int = 512,
    ):
        self.device = device
        self.G = copy.deepcopy(generator).eval().requires_grad_(False).to(device)
        vgg_url = (
            "https://nvlabs-fi-cdn.nvidia.com/"
            "stylegan2-ada-pytorch/pretrained/metrics/vgg16.pt"
        )
        print("Загружается VGG16 для проекции...")
        with dnnlib.util.open_url(vgg_url, cache_dir=str(CACHE_DIR)) as file:
            self.vgg16 = torch.jit.load(file).eval().to(device)

        print("Вычисляются устойчивые статистики W...")
        random_state = np.random.RandomState(123)
        w_chunks = []
        generated = 0
        while generated < w_avg_samples:
            current_batch = min(
                statistics_batch_size,
                w_avg_samples - generated,
            )
            z = random_state.randn(
                current_batch,
                self.G.z_dim,
            ).astype(np.float32)
            z = torch.from_numpy(z).to(device)
            with torch.no_grad():
                current_w = self.G.mapping(z, None)[:, :1, :].float().cpu()
            w_chunks.append(current_w)
            generated += current_batch
            del z, current_w

        w_samples = torch.cat(w_chunks, dim=0).to(device)
        self.w_avg = w_samples.mean(dim=0, keepdim=True)
        self.w_std = torch.sqrt(
            ((w_samples - self.w_avg) ** 2).sum() / w_avg_samples
        )
        del w_chunks, w_samples
        gc.collect()
        torch.cuda.empty_cache()

    def _target_features(self, target_tensor: torch.Tensor) -> torch.Tensor:
        target_255 = (target_tensor + 1.0) * 127.5
        if target_255.shape[2] > 256:
            target_255 = F.interpolate(target_255, size=(256, 256), mode="area")
        with torch.no_grad():
            return self.vgg16(
                target_255,
                resize_images=False,
                return_lpips=True,
            )

    def _synthesis_losses(
        self,
        synthesized: torch.Tensor,
        target_tensor: torch.Tensor,
        target_features: torch.Tensor,
        pixel_weight: float,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        synthesized_255 = (synthesized + 1.0) * 127.5
        if synthesized_255.shape[2] > 256:
            synthesized_255 = F.interpolate(
                synthesized_255,
                size=(256, 256),
                mode="area",
            )
        synthesized_features = self.vgg16(
            synthesized_255,
            resize_images=False,
            return_lpips=True,
        )
        perceptual = (target_features - synthesized_features).square().sum()
        target_small = F.interpolate(
            target_tensor,
            size=(128, 128),
            mode="area",
        )
        synthesized_small = F.interpolate(
            synthesized,
            size=(128, 128),
            mode="area",
        )
        pixel = F.mse_loss(synthesized_small, target_small)
        objective = perceptual + pixel_weight * pixel
        return objective, perceptual, pixel

    @staticmethod
    def _noise_regularization(
        noise_buffers: Dict[str, torch.Tensor],
    ) -> torch.Tensor:
        if not noise_buffers:
            return torch.zeros([], device=DEVICE)
        regularization = torch.zeros(
            [],
            device=next(iter(noise_buffers.values())).device,
        )
        for noise in noise_buffers.values():
            current = noise[None, None, :, :]
            while True:
                regularization = regularization + (
                    current * torch.roll(current, shifts=1, dims=3)
                ).mean().square()
                regularization = regularization + (
                    current * torch.roll(current, shifts=1, dims=2)
                ).mean().square()
                if current.shape[2] <= 8:
                    break
                current = F.avg_pool2d(current, kernel_size=2)
        return regularization

    @staticmethod
    def _normalize_noise(noise_buffers: Dict[str, torch.Tensor]) -> None:
        with torch.no_grad():
            for noise in noise_buffers.values():
                noise -= noise.mean()
                noise *= noise.square().mean().clamp_min(1e-8).rsqrt()

    @staticmethod
    def _learning_rate(
        step: int,
        total_steps: int,
        initial_learning_rate: float,
    ) -> float:
        t = step / max(1, total_steps - 1)
        ramp_down = min(1.0, (1.0 - t) / 0.25)
        ramp_down = 0.5 - 0.5 * math.cos(ramp_down * math.pi)
        ramp_up = min(1.0, t / 0.05)
        return initial_learning_rate * ramp_down * ramp_up

    def project(
        self,
        image: Image.Image,
        num_steps: int = 700,
        w_only_steps: int = 180,
        initial_learning_rate: float = 0.06,
        initial_noise_factor: float = 0.035,
        pixel_weight: float = 10.0,
        noise_regularize_weight: float = 1e5,
        wplus_regularize_weight: float = 0.02,
        verbose: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        target_tensor = pil_to_model_tensor(
            image,
            self.G.img_resolution,
        ).unsqueeze(0).to(self.device)
        target_features = self._target_features(target_tensor)
        noise_buffers = {
            name: buffer
            for name, buffer in self.G.synthesis.named_buffers()
            if "noise_const" in name
        }

        with torch.no_grad():
            for noise in noise_buffers.values():
                noise.zero_()

        w_only_steps = min(max(20, w_only_steps), max(20, num_steps - 20))
        w_single = self.w_avg.detach().clone().requires_grad_(True)
        w_optimizer = torch.optim.Adam(
            [w_single],
            lr=initial_learning_rate,
            betas=(0.9, 0.999),
        )
        first_progress = tqdm(
            range(w_only_steps),
            disable=not verbose,
            leave=False,
            desc="W",
        )

        for step in first_progress:
            lr = self._learning_rate(
                step,
                w_only_steps,
                initial_learning_rate,
            )
            w_optimizer.param_groups[0]["lr"] = lr
            w_optimizer.zero_grad(set_to_none=True)
            ws = w_single.repeat(1, self.G.num_ws, 1)
            synthesized = self.G.synthesis(
                ws,
                noise_mode="const",
                force_fp32=True,
            )
            objective, perceptual, pixel = self._synthesis_losses(
                synthesized,
                target_tensor,
                target_features,
                pixel_weight,
            )
            objective.backward()
            w_optimizer.step()
            if verbose and step % 40 == 0:
                first_progress.set_postfix({
                    "lpips": f"{float(perceptual.detach()):.3f}",
                    "pixel": f"{float(pixel.detach()):.4f}",
                })

        anchor_w = w_single.detach().repeat(1, self.G.num_ws, 1)
        optimized_w = anchor_w.clone().requires_grad_(True)
        with torch.no_grad():
            for noise in noise_buffers.values():
                noise.copy_(torch.randn_like(noise))
                noise.requires_grad_(True)

        optimizer = torch.optim.Adam(
            [optimized_w] + list(noise_buffers.values()),
            lr=initial_learning_rate,
            betas=(0.9, 0.999),
        )
        refinement_steps = num_steps - w_only_steps
        best_score = float("inf")
        best_w = optimized_w.detach().clone()
        best_noise = {
            name: noise.detach().clone()
            for name, noise in noise_buffers.items()
        }
        progress = tqdm(
            range(refinement_steps),
            disable=not verbose,
            leave=False,
            desc="W+",
        )

        for step in progress:
            t = step / max(1, refinement_steps - 1)
            lr = self._learning_rate(
                step,
                refinement_steps,
                initial_learning_rate,
            )
            for group in optimizer.param_groups:
                group["lr"] = lr

            optimizer.zero_grad(set_to_none=True)
            noise_scale = (
                self.w_std
                * initial_noise_factor
                * max(0.0, 1.0 - t / 0.75) ** 2
            )
            noisy_w = optimized_w + torch.randn_like(optimized_w) * noise_scale
            synthesized = self.G.synthesis(
                noisy_w,
                noise_mode="const",
                force_fp32=True,
            )
            objective, perceptual, pixel = self._synthesis_losses(
                synthesized,
                target_tensor,
                target_features,
                pixel_weight,
            )
            noise_regularization = self._noise_regularization(noise_buffers)
            wplus_regularization = (
                optimized_w - anchor_w
            ).square().mean()
            loss = (
                objective
                + noise_regularize_weight * noise_regularization
                + wplus_regularize_weight * wplus_regularization
            )
            loss.backward()
            optimizer.step()
            self._normalize_noise(noise_buffers)

            score = float(objective.detach().cpu())
            if score < best_score:
                best_score = score
                best_w = optimized_w.detach().clone()
                best_noise = {
                    name: noise.detach().clone()
                    for name, noise in noise_buffers.items()
                }

            if verbose and step % 50 == 0:
                progress.set_postfix({
                    "lpips": f"{float(perceptual.detach()):.3f}",
                    "pixel": f"{float(pixel.detach()):.4f}",
                    "best": f"{best_score:.3f}",
                })

        with torch.no_grad():
            for name, noise in noise_buffers.items():
                noise.copy_(best_noise[name])
                noise.requires_grad_(False)
            reconstruction = self.G.synthesis(
                best_w,
                noise_mode="const",
                force_fp32=True,
            )
        return best_w, reconstruction

## 19. Загрузка и точное выравнивание реальных фотографий

Загрузите фронтальный портрет одного человека. MTCNN найдёт лицо и выровняет глаза, нос и рот под геометрию FFHQ. В предпросмотре обязательно проверьте, что лицо не обрезано и находится ровно по центру.

## 20. Сохранение результатов на Google Drive

In [ ]:
if CFG.save_to_google_drive:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/stylegan_nada_anime_project")
    drive_dir.mkdir(parents=True, exist_ok=True)
    paths_to_copy = [
        FINAL_MODEL_PATH,
        history_path,
        curves_path,
        random_pair_path,
        validation_grid,
        metrics_path,
        VALIDATION_DIR / "aligned_preview.png",
        VALIDATION_DIR / "alignment_records.json",
        VALIDATION_DIR / "photo_reconstruction_anime.png",
        VALIDATION_DIR / "photo_results.json",
    ]
    if final_checkpoint is not None:
        paths_to_copy.append(final_checkpoint)

    for source_path in paths_to_copy:
        source_path = Path(source_path)
        if source_path.exists():
            destination = drive_dir / source_path.name
            shutil.copy2(source_path, destination)
            print("Сохранено:", destination)
else:
    print("Сохранение на Google Drive отключено.")

## 21. Архив всех результатов

In [ ]:
from google.colab import files

export_dir = Path("/content/stylegan_nada_full_export")
shutil.rmtree(export_dir, ignore_errors=True)
export_dir.mkdir(parents=True, exist_ok=True)

for folder_name in [
    "models",
    "logs",
    "training_samples",
    "validation",
    "photo_results",
]:
    source_dir = OUTPUT_ROOT / folder_name
    if source_dir.exists():
        shutil.copytree(
            source_dir,
            export_dir / folder_name,
            dirs_exist_ok=True,
        )

archive_path = shutil.make_archive(
    "/content/stylegan_nada_anime_full_result",
    "zip",
    root_dir=export_dir,
)
print("Архив без кэша моделей:", archive_path)
files.download(archive_path)

## 22. Gradio-сервис только для инференса

Сервис автоматически выравнивает лицо, один раз выполняет двухэтапную проекцию и возвращает три варианта стилизации. Обучение и изменение исходных весов в веб-сервисе не выполняются.

In [ ]:
import gradio as gr

def gradio_anime_conversion(
    input_image: Image.Image,
    projection_steps: int,
):
    if input_image is None:
        raise gr.Error("Загрузите изображение.")

    prepared_image, alignment_info = align_face_ffhq(
        input_image,
        CFG.model_resolution,
    )
    projected_w, reconstruction = projector.project(
        prepared_image,
        num_steps=int(projection_steps),
        w_only_steps=min(CFG.projection_w_steps, int(projection_steps) // 3),
        verbose=False,
    )
    outputs = []

    for strength in CFG.inference_strengths:
        set_style_strength(strength)
        with torch.no_grad():
            anime = G_stylized.synthesis(
                projected_w,
                noise_mode="const",
                force_fp32=True,
            )
        outputs.append(image_tensor_to_pil(anime[0]))

    return (
        prepared_image,
        image_tensor_to_pil(reconstruction[0]),
        *outputs,
        json.dumps(alignment_info, ensure_ascii=False),
    )

if CFG.launch_gradio:
    output_components = [
        gr.Image(type="pil", label="Выровненный вход"),
        gr.Image(type="pil", label="Реконструкция StyleGAN"),
    ] + [
        gr.Image(type="pil", label=f"Аниме, сила {strength:.2f}")
        for strength in CFG.inference_strengths
    ] + [
        gr.Textbox(label="Alignment"),
    ]

    demo = gr.Interface(
        fn=gradio_anime_conversion,
        inputs=[
            gr.Image(type="pil", label="Фотография человека"),
            gr.Slider(
                minimum=400,
                maximum=1000,
                value=CFG.projection_steps,
                step=50,
                label="Шагов проекции",
            ),
        ],
        outputs=output_components,
        title="StyleGAN-NADA: фото → аниме",
        description=(
            "Сначала лицо выравнивается по пяти точкам. "
            "Затем показываются мягкий, стандартный и сильный варианты."
        ),
    )
    demo.queue(default_concurrency_limit=1).launch(
        share=True,
        debug=False,
    )
else:
    print("Gradio отключён в конфигурации.")

## Что приложить к отчёту

Зафиксируйте выбранные блоки, число обучаемых параметров, гиперпараметры, графики loss, сетки `source | anime`, значения валидационных метрик, результаты на реальных фотографиях и примеры ошибок. В выводах опишите компромисс между силой стилизации и сохранением геометрии лица, а также зависимость метода от CLIP-промптов и домена исходного StyleGAN.